In [1]:
import os
import sys

sys.path.append(os.path.join(os.getcwd(), "preprocessing", "label_mapping"))
from label_mapping import build_labeled_dataset

# mapping the label
df = build_labeled_dataset()


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26179 entries, 0 to 26178
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   image_path  26179 non-null  object
 1   label_it    26179 non-null  object
 2   label_en    26179 non-null  object
dtypes: object(3)
memory usage: 613.7+ KB


In [ ]:
sys.path.append(os.path.join(os.getcwd(), "preprocessing", "data_loader"))
from data_loader import build_train_val_test_generators

# ImageDataGenerator resizes each image on load and rescales pixels to [0, 1]
# (normalization). 128x128 instead of 224 to train much faster on CPU, at a
# small accuracy cost. 70/15/15 train/val/test split.
train_generator, val_generator, test_generator = build_train_val_test_generators(
    df, project_root=os.getcwd(), image_size=(128, 128)
)

print(
    f"Train batches: {len(train_generator)}, "
    f"Val batches: {len(val_generator)}, "
    f"Test batches: {len(test_generator)}"
)
train_generator.class_indices

In [ ]:
sys.path.append(os.path.join(os.getcwd(), "model"))
from cnn_model import init_cnn_model

# input_shape and num_classes both read from the generator so the model always
# stays in sync with the data (image_shape is e.g. (128, 128, 3)).
model = init_cnn_model(
    input_shape=train_generator.image_shape,
    num_classes=len(train_generator.class_indices),
)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop once val_loss stops improving for 3 epochs, and restore the weights
# from the best epoch so we don't keep an overfitted final state.
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stopping],
)